# Tutorial: Material Parameters Definition

This notebook isolates the sample material stack, recipe parsing, refractive indices, and dielectric tensor construction.

The recipe string is compact but powerful: slashes create separate propagated layers, adjacent materials without a slash create one effective-medium layer, and `[ ... ]xN` repeats layer blocks.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Make the notebook runnable from a fresh clone without requiring an editable install.
repo_root = Path.cwd()
if (repo_root / "src").exists() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

try:
    # Use the interactive widget backend when it is available in JupyterLab.
    %matplotlib widget
except Exception:
    # Plain scripts and some notebook renderers do not understand IPython magics.
    pass

plt.rcParams["figure.constrained_layout.use"] = True

from scattering_calculator.simulation_pipelines import simulation_configuration as sim
from scattering_calculator.sample_generator import pattern_generator

from scattering_calculator.sample_generator import structures


## 1. Define the X-ray energy and recipe

Material optical constants are energy-dependent, so the X-ray energy must be set before the sample is built.


In [ ]:
xray_config = sim.XRayConfig(
    energy=778.0,
    photon_flux=1e10,
    pol="CR",
    coherence_length=(10e-6, 10e-6),
)
xray_config.setup()

recipe = "[Au(80)/Cr(5)]x3/SiN(80)/Pt(4)Co(6)/Pt(2)"
sample_shape = [0, 768, 768]       # first entry 0 means: fill in number of layers after setup
real_space_pixel_size = 4e-9


## 2. Parse the recipe before building the sample

This is the safest way to teach what the string means.


In [ ]:
parsed_recipe = structures.parse_recipe(recipe, sample_name="didactic multilayer")

for i, layer in enumerate(parsed_recipe.layers):
    if layer.components is None:
        detail = layer.material
    else:
        detail = " + ".join(f"{name}({thickness * 1e9:g} nm)" for name, thickness in layer.components)
    print(f"{i:02d}: {layer.material:20s} thickness={layer.thickness * 1e9:7.2f} nm   {detail}")


## 3. Build the `SampleConfig`

`SampleConfig.setup()` loads material parameters and constructs the layered `Structure`.


In [ ]:
sample_config = sim.SampleConfig(
    recipe=recipe,
    sample_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    xray_config=xray_config,
    sample_name="material-parameter tutorial",
)
sample_config.setup()
structure = sample_config.sample_structure

print("sample shape after setup:", structure.sample_shape)
print("layer names:", structure.layer_names)
print("layer thicknesses in nm:", np.array(structure.layer_thicknesses) * 1e9)


## 4. Inspect refractive index channels

The stored refractive index has three channels:

- `n0`: isotropic charge response;
- `dn_c`: circular magnetic contribution, used for XMCD;
- `dn_l`: linear magnetic contribution, used for XMLD.


In [ ]:
indices = np.asarray(structure.layer_refractive_indices)

print(f"{'layer':>2s} {'name':>18s} {'n0 real':>14s} {'n0 imag':>14s} {'dn_c abs':>14s} {'dn_l abs':>14s}")
for i, (name, n) in enumerate(zip(structure.layer_names, indices)):
    print(f"{i:2d} {name:>18s} {n[0].real:14.7g} {n[0].imag:14.7g} {abs(n[1]):14.7g} {abs(n[2]):14.7g}")


## 5. Visualize the layer stack

This custom plot colors layers by the isotropic `n0` channel. The material database also provides magnetic channels (`dn_c`, `dn_l`), which are inspected numerically above and enter the dielectric tensor below.


In [ ]:
n0 = np.asarray([n[0] for n in structure.layer_refractive_indices])
thickness_nm = np.asarray(structure.layer_thicknesses) * 1e9

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, values, title in zip(
    axes,
    [np.real(n0), np.imag(n0)],
    ["real n0", "imag n0"],
):
    y_bottom = 0.0
    norm = plt.Normalize(values.min(), values.max() + 1e-12)
    for name, thickness, value in zip(structure.layer_names, thickness_nm, values):
        y_center = y_bottom + thickness / 2
        ax.barh(y_center, 1, height=thickness, color=plt.cm.viridis_r(norm(value)), edgecolor="black")
        ax.text(0.5, y_center, name, ha="center", va="center", fontsize=8, color="white")
        y_bottom += thickness
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_ylabel("thickness in nm")
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.viridis_r), ax=ax, label=title)


## 6. Build a simple magnetization and the shared FTH aperture mask

The final dielectric tensor requires both:

- `magnetization`, shape `(Nz, Ny, Nx, 3)`;
- `mask`, shape `(Nz, Ny, Nx)`, where 1 is material and 0 is vacuum/open aperture. Here we use the same FTH aperture geometry as the multislice/ROI tutorial.

For this material tutorial we use a saturated magnetic pattern and a fully closed mask so that the material parameters are the main thing changing the tensor.


In [ ]:
nz, ny, nx = structure.sample_shape
mz = np.ones((ny, nx))
magnetization = pattern_generator.map_magnetization_to_3d(
    magnetic_pattern_x=np.zeros_like(mz),
    magnetic_pattern_y=np.zeros_like(mz),
    magnetic_pattern_z=mz,
    nr_repeats=nz,
)
sample_config.assign_magnetic_pattern(magnetization)
membrane_index = structure.layer_names.index("SiN")
aperture_config = sim.FrontApertureConfig(
    aperture_method="FTH_circular",
    aperture_shape=(nz, ny, nx),
    real_space_pixel_size=real_space_pixel_size,
    aperture_thicknesses=structure.layer_thicknesses,
    aperture_layer_names=structure.layer_names,
    aperture_config={
        "apertures_type": ["OH", "RH", "RH"],
        "apertures_radius": [555e-9, 100e-9, 34e-9],
        "apertures_center": [(0.0, 0.0), (-1240e-9, -1225e-9), (1225e-9, -1200e-9)],
        "apertures_sigma": [4e-9, 2e-9, 2e-9],
        "apertures_angle": [0.0, 0.0, 0.0],
        "apertures_ellipticity": [1.0, 1.0, 1.0],
        "apertures_roughness": [0.0, 0.02, 0.02],
        "apertures_roughness_modes": [(0, 0), (3, 10), (3, 10)],
        "apertures_seed": [1, 2, 3],
        "apertures_top_radius_factor": [1.3, 1.5, 1.75],
        "aperture_taper_depth": float(np.sum(structure.layer_thicknesses[:max(0, membrane_index - 2)])),
        "thickness_OH": float(np.sum(structure.layer_thicknesses[:membrane_index])),
    },
    use_roi=True,
)
aperture_config.setup()
sample_config.assign_aperture_mask(aperture_config.return_aperture())

structure.calculate_final_dielectric_tensor(
    use_aperture_roi=True,
    compact=True,
    beam_direction=None,  # normal-incidence Jones basis; pass (kx, ky, kz) for projected magnetic contrast
)
eps_stack = structure.final_dielectric_tensor

print("compact dielectric tensor stack shape:", eps_stack.shape)
print("number of per-layer ROI patches:", [len(patches) for patches in eps_stack.patches])


## 7. Materialize one layer for inspection

Compact mode avoids allocating the whole dense stack, but you can materialize one layer when you want to plot or debug it.


In [ ]:
layer_index = structure.layer_names.index("Pt(4)Co(6)") if "Pt(4)Co(6)" in structure.layer_names else len(structure.layer_names) - 1
eps_layer = eps_stack.materialize_layer(layer_index)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
panels = [
    (np.real(eps_layer[..., 0, 0]), "real eps_xx"),
    (np.imag(eps_layer[..., 0, 0]), "imag eps_xx"),
    (np.imag(eps_layer[..., 0, 1]), "imag eps_xy"),
]
for ax, (data, title) in zip(axes, panels):
    im = ax.imshow(data)
    ax.set_title(f"{structure.layer_names[layer_index]}: {title}")
    ax.set_axis_off()
    fig.colorbar(im, ax=ax)
